In [1]:
#!pip uninstall -y scipy scikit-learn

# The '--no-cache-dir' flag ensures we download fresh copies
# The '--force-reinstall' flag reinstalls even if it thinks a version is already there
#!pip install --no-cache-dir --force-reinstall scikit-learn

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score

# Import the custom modules from your project
# Ensure these files are in the same directory as your notebook
from datasets import ecoli
from utils import split_data, split_xy

# Load the ecoli dataset and the model structure
# The 'model' object is an untrained instance of the algorithm
# The 'data' object holds the feature and target variables
model, data = ecoli()

print("Setup complete. Ecoli dataset is loaded.")

In [ ]:
# 1. Split the data into a training set (80%) and a testing set (20%)
data_train, data_test = split_data(data, ratio=0.8)
X_test, Y_test = split_xy(data_test)

# 2. Instantiate a fresh model to train
confold_model, _ = ecoli()

# 3. Train the CON-FOLD model on the training data
# We use the optimized parameters from your notebook for the Ecoli dataset
confold_model.confidence_fit(data_train, improvement_threshold=0.08)
confold_model.rules = prune_rules(confold_model.rules, confidence=0.6)
confold_model.asp() # This method likely finalizes the rule set

# 4. Print the generated rules
print("--- Generated CON-FOLD Rules ---")
if hasattr(confold_model, 'crs') and confold_model.crs:
    for i, rule in enumerate(confold_model.crs):
        # The rule format appears to be [body, head, confidence]
        # We check if confidence is > 0 as done in your notebook
        if rule[-1] > 0:
            print(f"Rule {i+1}: IF {rule[0]} THEN {rule[1]} (Confidence: {rule[-1]:.4f})")
else:
    print("No rules were generated or the attribute 'crs' is not available.")

# 5. Make predictions and evaluate accuracy on the test set
# The predict method returns a list of tuples (predicted_class, confidence)
predictions_with_confidence = confold_model.predict(X_test)
# Extract just the predicted class for accuracy calculation
predictions = [p[0] for p in predictions_with_confidence]

accuracy = accuracy_score(Y_test, predictions)
print("\n--- Model Performance ---")
print(f"Accuracy on the single test set: {accuracy:.4f}")